In [45]:
from pyspark.sql import SparkSession

In [46]:
spark = SparkSession.builder.master('local[*]').appName('test').getOrCreate()

25/07/23 22:58:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/07/23 22:58:14 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
spark.getActiveSession()

In [4]:
df = spark.read.parquet('fhvhv/2021/01')

since we saved the original csv as a parquet, the parquet contains the metadata. when we read the parquet, it automatically has the original schema

In [7]:
df.select('pickup_datetime', 'dropoff_datetime', 'pulocationid', 'dolocationid')\
    .filter(df.hvfhs_license_num == 'HV0003')\
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|pulocationid|dolocationid|
+-------------------+-------------------+------------+------------+
|2021-01-01 14:19:18|2021-01-01 14:51:27|         229|         132|
|2021-01-02 21:42:11|2021-01-02 21:52:29|         231|         246|
|2021-01-01 02:59:54|2021-01-01 03:13:00|          85|          35|
|2021-01-01 00:35:57|2021-01-01 01:01:29|          40|         129|
|2021-01-01 01:59:32|2021-01-01 02:11:47|         263|           7|
|2021-01-01 01:45:04|2021-01-01 01:51:49|          37|          36|
|2021-01-01 03:41:19|2021-01-01 03:49:23|         232|         148|
|2021-01-01 02:36:17|2021-01-01 02:44:45|         167|         168|
|2021-01-03 15:18:05|2021-01-03 15:39:45|         188|          37|
|2021-01-04 07:17:45|2021-01-04 07:22:03|         216|         216|
|2021-01-01 19:28:55|2021-01-01 19:34:47|          36|          36|
|2021-01-01 00:30:05|2021-01-01 00:40:31|       

In [38]:
import multiprocessing
print(multiprocessing.cpu_count())  # same as what local[*] uses
spark.sparkContext.defaultParallelism

8


8

In [26]:
import pandas as pd

df_yellow_pd = pd.read_csv('./data/raw/yellow/2021/01/yellow_tripdata_2021_01.csv.gz', nrows = 1000)
print(df_yellow_pd.dtypes)

df_yellow = spark.createDataFrame(df_yellow_pd)
df_yellow.schema



VendorID                   int64
tpep_pickup_datetime      object
tpep_dropoff_datetime     object
passenger_count            int64
trip_distance            float64
RatecodeID                 int64
store_and_fwd_flag        object
PULocationID               int64
DOLocationID               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
congestion_surcharge     float64
dtype: object


StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', StringType(), True), StructField('tpep_dropoff_datetime', StringType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [49]:
from pyspark.sql import types

yellow_schema = types.StructType([
    types.StructField('VendorID', types.IntegerType(), True), 
    types.StructField('tpep_pickup_datetime', types.TimestampType(), True), 
    types.StructField('tpep_dropoff_datetime', types.TimestampType(), True), 
    types.StructField('passenger_count', types.IntegerType(), True), 
    types.StructField('trip_distance', types.DoubleType(), True), 
    types.StructField('RatecodeID', types.IntegerType(), True), 
    types.StructField('store_and_fwd_flag', types.StringType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('payment_type', types.IntegerType(), True), 
    types.StructField('fare_amount', types.DoubleType(), True), 
    types.StructField('extra', types.DoubleType(), True), 
    types.StructField('mta_tax', types.DoubleType(), True), 
    types.StructField('tip_amount', types.DoubleType(), True), 
    types.StructField('tolls_amount', types.DoubleType(), True), 
    types.StructField('improvement_surcharge', types.DoubleType(), True), 
    types.StructField('total_amount', types.DoubleType(), True), 
    types.StructField('congestion_surcharge', types.DoubleType(), True)
    ]
)

year = 2021
for month in range(1, 13) :
    try:
        print(f'processing data for {year}/{month}')
        input_path = f'./data/raw/yellow/{year}/{month:02d}/'
        output_path = f'./data/pq/yellow/{year}/{month:02d}/'

        df_yellow = spark.read\
                    .option('header', 'true')\
                    .schema(yellow_schema)\
                    .csv(input_path)

        df_yellow\
            .repartition(4)\
            .write.mode('overwrite')\
            .parquet(output_path)
    except Exception:
        break    


processing data for 2021/1


processing data for 2021/2


processing data for 2021/3


processing data for 2021/4


processing data for 2021/5


processing data for 2021/6


processing data for 2021/7


processing data for 2021/8


25/07/23 23:07:12 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ./data/raw/yellow/2021/08/.
java.io.FileNotFoundException: File data/raw/yellow/2021/08 does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Re

In [50]:
year = 2020
for month in range(1, 13) :
    print(f'processing data for {year}/{month}')
    input_path = f'./data/raw/yellow/{year}/{month:02d}/'
    output_path = f'./data/pq/yellow/{year}/{month:02d}/'

    df_yellow = spark.read\
                .option('header', 'true')\
                .schema(yellow_schema)\
                .csv(input_path)

    df_yellow\
        .repartition(4)\
        .write.mode('overwrite')\
        .parquet(output_path)

processing data for 2020/1


processing data for 2020/2


processing data for 2020/3


processing data for 2020/4


processing data for 2020/5


...

processing data for 2020/8


processing data for 2020/9


processing data for 2020/10


processing data for 2020/11


processing data for 2020/12


In [51]:
import pandas as pd

df_green_pd = pd.read_csv('./data/raw/green/2021/01/green_tripdata_2021_01.csv.gz', nrows = 1000)
print(df_green_pd.dtypes)

df_green = spark.createDataFrame(df_green_pd)
df_green.schema



VendorID                   int64
lpep_pickup_datetime      object
lpep_dropoff_datetime     object
store_and_fwd_flag        object
RatecodeID                 int64
PULocationID               int64
DOLocationID               int64
passenger_count            int64
trip_distance            float64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
ehail_fee                float64
improvement_surcharge    float64
total_amount             float64
payment_type               int64
trip_type                  int64
congestion_surcharge     float64
dtype: object


StructType([StructField('VendorID', LongType(), True), StructField('lpep_pickup_datetime', StringType(), True), StructField('lpep_dropoff_datetime', StringType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('RatecodeID', LongType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('ehail_fee', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('payment_type', LongType(), True), StructField('trip_type', LongType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [52]:
green_schema = types.StructType([
    types.StructField('VendorID', types.IntegerType(), True), 
    types.StructField('lpep_pickup_datetime', types.TimestampType(), True), 
    types.StructField('lpep_dropoff_datetime', types.TimestampType(), True), 
    types.StructField('store_and_fwd_flag', types.StringType(), True), 
    types.StructField('RatecodeID', types.IntegerType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('passenger_count', types.IntegerType(), True), 
    types.StructField('trip_distance', types.DoubleType(), True), 
    types.StructField('fare_amount', types.DoubleType(), True), 
    types.StructField('extra', types.DoubleType(), True), 
    types.StructField('mta_tax', types.DoubleType(), True), 
    types.StructField('tip_amount', types.DoubleType(), True), 
    types.StructField('tolls_amount', types.DoubleType(), True), 
    types.StructField('ehail_fee', types.DoubleType(), True), 
    types.StructField('improvement_surcharge', types.DoubleType(), True), 
    types.StructField('total_amount', types.DoubleType(), True), 
    types.StructField('payment_type', types.IntegerType(), True), 
    types.StructField('trip_type', types.IntegerType(), True), 
    types.StructField('congestion_surcharge', types.DoubleType(), True)])

year = 2021
for month in range(1, 13) :
    try:    
        print(f'processing data for {year}/{month}')
        input_path = f'./data/raw/green/{year}/{month:02d}/'
        output_path = f'./data/pq/green/{year}/{month:02d}/'

        df_green = spark.read\
                    .option('header', 'true')\
                    .schema(green_schema)\
                    .csv(input_path)

        df_green\
            .repartition(4)\
            .write.mode('overwrite')\
            .parquet(output_path)
    except Exception:
        break

processing data for 2021/1
processing data for 2021/2
processing data for 2021/3
processing data for 2021/4
processing data for 2021/5
processing data for 2021/6
processing data for 2021/7
processing data for 2021/8


25/07/23 23:08:42 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ./data/raw/green/2021/08/.
java.io.FileNotFoundException: File data/raw/green/2021/08 does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Reso

In [ ]:
year = 2020
for month in range(1, 13) :
    print(f'processing data for {year}/{month}')
    input_path = f'./data/raw/green/{year}/{month:02d}/'
    output_path = f'./data/pq/green/{year}/{month:02d}/'

    df_green = spark.read\
                .option('header', 'true')\
                .schema(green_schema)\
                .csv(input_path)

    df_green\
        .repartition(4)\
        .write.mode('overwrite')\
        .parquet(output_path)

processing data for 2020/1


processing data for 2020/2


processing data for 2020/3


processing data for 2020/4
processing data for 2020/5
processing data for 2020/6
processing data for 2020/7
processing data for 2020/8
processing data for 2020/9
processing data for 2020/10
processing data for 2020/11
processing data for 2020/12


25/07/24 02:39:30 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1002796 ms exceeds timeout 120000 ms
25/07/24 02:39:30 WARN SparkContext: Killing executors is not supported by current scheduler.
25/07/24 02:39:31 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at 

In [44]:
spark.stop()